In [ ]:
import os

# ── Configuration ───────────────────────────────────────────────────────────
# Inputs are the per-fraction d_exact tables written by score_subsets
# (scripts/slurm/run_score_subsets.sh, one sbatch per FRACTION), which store one
# row per (seed, method, K): fraction, test accuracy, D, ...
SCRATCH = "/network/scratch/a/ahmedm"
DATASET = "cifar10"
ARCHITECTURES = ["resnet18", "vgg16", "inception"]
FRACTIONS = [0.1, 0.2, 0.4, 0.5]        # which fraction-tagged CSVs to merge
KL_K = 1                                # checkpoint count of "ours" (the paper's run)
KL_NAME = f"Gradient Swap, K={KL_K} (ours)"
OUT_DIR = os.path.join(SCRATCH, "data_selection_baselines", "plots")


In [ ]:
import glob
import os

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

tol_bright = ['#4477AA', '#EE6677', '#228833', '#AA3377', '#66CCEE', '#CCBB44', '#000000']
linestyles = ['-', '--', '-.', ':', '-', '--', '-.']
markers = ["*", "o", "s", "^", "v", "D", "<", ">"]

# ---------------------------------------------------------------- journal style
plt.rcParams.update({
    # fonts: serif (Times-like), embedded as TrueType so journals accept the PDF
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "STIXGeneral", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,

    # font sizes (bigger)
    "font.size": 22,
    "axes.labelsize": 28,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "legend.fontsize": 20,

    # axes / ticks
    "axes.linewidth": 1.8,
    "axes.labelpad": 8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 7,
    "xtick.major.width": 1.8,
    "xtick.major.pad": 6,
    "legend.frameon": True,
    "legend.fancybox": False,
    "legend.edgecolor": "0.2",
    "legend.framealpha": 0.95,
    "legend.handlelength": 2.6,
    "legend.borderpad": 0.5,
    "legend.labelspacing": 0.35,

    # output
    "savefig.dpi": 300,
})

os.makedirs(OUT_DIR, exist_ok=True)
baselines_root = os.path.join(SCRATCH, "data_selection_baselines")
REQUIRED = {"fraction", "method", "num_checkpoints_tag", "seed", "acc"}

for architecture in ARCHITECTURES:
    # ------------------------------------------------------------- load & merge
    # score_subsets tags single-fraction runs (d_exact_<ds>_<arch>_f<f>.csv);
    # merge the four fraction tables into one per architecture.
    csvs = sorted(glob.glob(os.path.join(baselines_root, f"d_exact_{DATASET}_{architecture}_f*.csv")))
    if not csvs:
        raise FileNotFoundError(f"no d_exact CSVs for {DATASET}/{architecture} under {baselines_root}")
    print(f"{architecture}: " + ", ".join(os.path.basename(p) for p in csvs))
    df = pd.concat([pd.read_csv(p) for p in csvs], ignore_index=True)
    missing = REQUIRED - set(df.columns)
    if missing:
        raise ValueError(f"{csvs[0]} lacks columns {sorted(missing)} -- stale pre-fix CSV?")
    df = df[df["fraction"].isin(FRACTIONS)]
    df = df.dropna(subset=["acc"])

    # keep the KL run at KL_K checkpoints, keep all baselines (no K tag)
    is_kl = df["method"].str.startswith("kl_faithful")
    df = df[~is_kl | (df["num_checkpoints_tag"] == KL_K)].copy()
    if (df["method"].str.startswith("kl_faithful")).sum() == 0:
        print(f"  WARNING: no kl_faithful rows with num_checkpoints_tag == {KL_K}")
    df.loc[df["method"].str.startswith("kl_faithful"), "method"] = KL_NAME
    df.loc[df["method"].str.startswith("facility_location"), "method"] = "Facility Location"
    df.loc[df["method"].str.startswith("graphcut"), "method"] = "GraphCut"
    df.loc[df["method"].str.startswith("herding"), "method"] = "Herding"
    df.loc[df["method"].str.startswith("moderate"), "method"] = "Moderate Coreset"
    df.loc[df["method"].str.startswith("moso"), "method"] = "MoSo"
    df.loc[df["method"].str.startswith("random"), "method"] = "Random"
    df.loc[df["method"].str.startswith("el2n"), "method"] = "EL2N"
    df["frac"] = df["fraction"]          # the scorer records the pool-relative fraction
    # 'acc' is the final-epoch test accuracy in % (the old notebook's
    # budget/TRAIN_SIZE fraction and 100-error columns no longer exist)

    # drop EL2N's first (smallest-fraction) point
    el2n_first = df.loc[df["method"] == "EL2N", "frac"].min()
    if pd.notna(el2n_first):
        df = df[~((df["method"] == "EL2N") & (df["frac"] == el2n_first))]

    # ------------------------------------------------------------- aggregate
    agg = (df.groupby(["method", "frac"])["acc"]
             .agg(mean="mean", std="std", n="count")
             .reset_index())
    agg["std"] = agg["std"].fillna(0.0)
    print(agg.to_string(index=False))        # n = number of seeds per point

    # ------------------------------------------------------------- style
    # ours gets slot 0; baselines follow, with EL2N last (it is mostly off-plot)
    others = [m for m in sorted(agg["method"].unique()) if m not in (KL_NAME, "EL2N")]
    if "EL2N" in set(agg["method"]):
        others.append("EL2N")
    order = [KL_NAME] + others

    def get_style(idx):
        # 8 curves but 7 colors: on wrap-around, shift the linestyle so that
        # (color, linestyle) pairs stay unique
        return dict(color=tol_bright[idx % 7],
                    ls=linestyles[(idx + idx // 7) % 7],
                    marker=markers[idx % len(markers)])

    fracs = sorted(agg["frac"].unique())

    # ------------------------------------------------------------- plot
    fig, ax = plt.subplots(figsize=(10, 7.5))

    for idx, m in enumerate(order):
        d = agg[agg["method"] == m].sort_values("frac")
        s = get_style(idx)
        is_ours = (m == KL_NAME)
        ax.plot(d["frac"], d["mean"], color=s["color"], linestyle=s["ls"],
                marker=s["marker"], lw=3.4 if is_ours else 2.6,
                ms=18 if is_ours else 10, label=m,
                markeredgecolor=s["color"], zorder=10 if is_ours else 3)
        ax.fill_between(d["frac"], d["mean"] - d["std"], d["mean"] + d["std"],
                        color=s["color"], alpha=0.15, lw=0, zorder=2)

    # y-limits set from everything except EL2N, so EL2N runs off the bottom
    ref = agg[agg["method"] != "EL2N"]
    lo = (ref["mean"] - ref["std"]).min()
    hi = (ref["mean"] + ref["std"]).max()
    pad = 0.06 * (hi - lo)
    ax.set_ylim(lo - pad, hi + pad)

    ax.set_xticks(fracs)
    ax.set_xlim(min(fracs) - 0.04, max(fracs) + 0.04)
    ax.set_xlabel("Fraction of the training set")
    ax.set_ylabel("Test accuracy (%)")
    ax.grid(True, linestyle="--", linewidth=0.8, alpha=0.4)
    ax.set_axisbelow(True)
    ax.legend(loc="lower right")

    fig.tight_layout()
    out_pdf = os.path.join(OUT_DIR, f"{DATASET}_{architecture}_accuracy.pdf")
    fig.savefig(out_pdf, bbox_inches="tight", pad_inches=0.05)
    plt.close(fig)
    print(f"Saved {out_pdf}")
